In [1]:
# The following code is from the white box lab.

# Initial Setup and Dependencies

import json
import pickle
import random
import re
import urllib.request
import zipfile
from pathlib import Path
import numpy as np

# Reproducibility
random.seed(1337)
np.random.seed(1337)

In [2]:
# Import data science and machine learning libraries:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

In [3]:
# We'll also import the HTB library utilities for consistent styling and helper functions:

from htb_ai_library import (
    AZURE,
    HACKER_GREY,
    HTB_GREEN,
    MALWARE_RED,
    NODE_BLACK,
    NUGGET_YELLOW,
    WHITE,
    AQUAMARINE,
    load_model,
    save_model,
)

In [4]:
# Loading the SMS Spam Dataset
# We'll use the SMS Spam Collection dataset from the UCI Machine Learning Repository, which contains 5,574 labeled messages. To avoid unnecessary downloads, implement a caching mechanism that checks for local copies first:

print("\n[*] Loading SMS Spam Dataset...")

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
dataset_path = data_dir / "sms_spam.csv"



[*] Loading SMS Spam Dataset...


In [5]:
# Checking for cached data

if dataset_path.exists():
    print(f"[+] Using cached dataset: {dataset_path}")
    df = pd.read_csv(dataset_path)
else:
    print("[*] Downloading from UCI repository...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
    zip_path = data_dir / "sms_spam.zip"

    urllib.request.urlretrieve(url, zip_path)

[+] Using cached dataset: data/sms_spam.csv


In [6]:
## Extracting and Processing the Download:
#
#with zipfile.ZipFile(zip_path, "r") as zf:
#    with zf.open("SMSSpamCollection") as f:
#        lines = [line.decode("utf-8").strip() for line in f]
#
## Parse tab-separated format
#data = []
#for line in lines:
#    parts = line.split('\t')
#    if len(parts) == 2:
#        data.append({"label": parts[0].lower(), "message": parts[1]})
#
#df = pd.DataFrame(data)
#df.to_csv(dataset_path, index=False)
#zip_path.unlink()
#print(f"[+] Dataset saved to {dataset_path}")

In [7]:
# Understanding the Dataset
# Now let's explore the dataset structure and class distribution to understand what we're working with:

print(f"[+] Loaded {len(df)} messages")
print(f"    Spam: {sum(df['label'] == 'spam')}")
print(f"    Ham: {sum(df['label'] == 'ham')}")

[+] Loaded 5574 messages
    Spam: 747
    Ham: 4827


In [8]:
# Checking some examples from the dataset:

print("\n[*] Sample messages:")
print("\nSPAM samples:")
for msg in df[df['label'] == 'spam']['message'].head(3):
    print(f"  - {msg[:80]}...")
print("\nHAM samples:")
for msg in df[df['label'] == 'ham']['message'].head(3):
    print(f"  - {msg[:80]}...")


[*] Sample messages:

SPAM samples:
  - Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 8...
  - FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some...
  - WINNER!! As a valued network customer you have been selected to receivea £900 pr...

HAM samples:
  - Go until jurong point, crazy.. Available only in bugis n great world la e buffet...
  - Ok lar... Joking wif u oni......
  - U dun say so early hor... U c already then say......


In [9]:
# Starting the Preprocessing part
# Minimal cleaning and analysis

import html as html_module
import unicodedata

In [10]:
# We implement a minimal cleaning function that preserves spam indicators while normalizing encoding and whitespace:

def minimal_clean(text):
    """
    Minimal cleaning that preserves spam indicators.

    Parameters: text (str) raw SMS message
    Returns: str cleaned text with entities decoded, unicode normalized, and
             whitespace collapsed while keeping informative symbols.
    """
    # Decode HTML entities (e.g., &amp; -> &)
    text = html_module.unescape(text)

    # Normalize unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Clean up excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\r+', ' ', text)

    return text.strip()

In [11]:
# Final cleaning for Vectorization

def clean_text(text):
    """
    Final cleaning for vectorization.

    Converts to lowercase and removes only problematic characters so that
    informative symbols remain available to the vectorizer.

    Parameters:
        text (str): Preprocessed message from `minimal_clean`.

    Returns:
        str: Normalized, whitespace‑collapsed text ready for tokenization.
    """
    text = text.lower()
    # Keep numbers, currency symbols, punctuation - they're spam features!
    # Only remove truly problematic characters
    text = re.sub(r'[^\w\s£$€¥!?.,;:\'\"-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [12]:
# Applying Preprocessing

print("[*] Applying minimal text cleaning (preserving spam indicators)...")
df['preprocessed'] = df['message'].apply(minimal_clean)

# Apply final cleaning for vectorization
df['clean_message'] = df['preprocessed'].apply(clean_text)

[*] Applying minimal text cleaning (preserving spam indicators)...


In [13]:
# Inspect what preprocessing preserved

print("\n[*] Sample spam messages with preserved features:")
spam_samples = df[df['label'] == 'spam'].sample(3, random_state=42)
for idx, row in spam_samples.iterrows():
    msg = row['preprocessed'][:100] + "..." if len(row['preprocessed']) > 100 else row['preprocessed']
    print(f"  - {msg}")


[*] Sample spam messages with preserved features:
  - Summers finally here! Fancy a chat or flirt with sexy singles in yr area? To get MATCHED up just rep...
  - This is the 2nd time we have tried 2 contact u. U have won the 750 Pound prize. 2 claim is easy, cal...
  - Get ur 1st RINGTONE FREE NOW! Reply to this msg with TONE. Gr8 TOP 20 tones to your phone every week...


In [14]:
# Removing Duplicates and Empty Messages

# Remove only exact duplicates
original_size = len(df)
df = df.drop_duplicates(subset=['label', 'clean_message'])
print(f"\n[+] Removed {original_size - len(df)} duplicates")

# Remove empty messages
before_empty = len(df)
df = df[df['clean_message'].str.len() > 0]
print(f"[+] Removed {before_empty - len(df)} empty messages")


[+] Removed 419 duplicates
[+] Removed 0 empty messages


In [15]:
# Creating Train and Test Sets
# Let's split the data into training and testing sets. Use stratified sampling to maintain the class distribution across both sets:

X = df['clean_message'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n[+] Data split:")
print(f"    Training: {len(X_train)} messages")
print(f"    Testing: {len(X_test)} messages")


[+] Data split:
    Training: 4124 messages
    Testing: 1031 messages


In [16]:
# Training the Naive Bayes Classifier
# We'll now train our MultinomialNB classifier with a count vectorizer for feature extraction. To ensure reproducibility, we'll also implement model persistence:

# Setting up model training

print("\n[*] Training Naive Bayes classifier...")

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "spam_classifier.pkl"

if model_path.exists():
    print(f"[+] Loading saved model from {model_path}")
    with open(model_path, 'rb') as f:
        saved_data = pickle.load(f)
        vectorizer = saved_data['vectorizer']
        classifier = saved_data['classifier']

    # Transform data using existing vocabulary
    X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
else:
    # Configure vectorizer to capture spam patterns
    vectorizer = CountVectorizer(
        max_features=3000,
        token_pattern=r'\b\w+\b|[£$€¥]+|\d+|!!+|\?\?+|\.\.+',
        lowercase=True,
        stop_words='english'
    )
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    classifier = MultinomialNB()
    classifier.fit(X_train_vec, y_train)

    # Save model for reproducibility
    with open(model_path, 'wb') as f:
        pickle.dump({'vectorizer': vectorizer, 'classifier': classifier}, f)
    print(f"[+] Model saved to {model_path}")


[*] Training Naive Bayes classifier...
[+] Loading saved model from models/spam_classifier.pkl


In [17]:
# Evaluating Model Performance
# Let's evaluate the classifier's performance on both training and test sets to check for overfitting:

# Calculate accuracy scores
train_acc = classifier.score(X_train_vec, y_train)
test_acc = classifier.score(X_test_vec, y_test)
print(f"[+] Training accuracy: {train_acc:.4f}")
print(f"[+] Testing accuracy: {test_acc:.4f}")

# Get detailed predictions
y_pred = classifier.predict(X_test_vec)
print("\n[*] Classification Report:")
print(classification_report(y_test, y_pred))

[+] Training accuracy: 0.9918
[+] Testing accuracy: 0.9855

[*] Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       903
        spam       0.95      0.93      0.94       128

    accuracy                           0.99      1031
   macro avg       0.97      0.96      0.97      1031
weighted avg       0.99      0.99      0.99      1031



In [18]:
# Black-Box Core Components
# The black-box attack scenario simulates realistic conditions where attackers have only query access to the target model. Without access to internal parameters, we must discover effective words through systematic exploration within a limited query budget. Build the core components that enable this discovery process.

# Attack Simulation Setup
print("\n[*] Simulating black-box attack scenario...")
print("[*] Budget: 1000 queries")

# Simulate limited query access
query_budget = 1000
queries_used = 0
query_log = []


[*] Simulating black-box attack scenario...
[*] Budget: 1000 queries


In [19]:
# Word Candidates
# Extracting Words from Ham Messages
def extract_ham_word_freq(X_train, y_train, sample_size=500):
    """
    Compute token frequencies from a sample of ham messages.

    Parameters
    ----------
    X_train : array-like of str
        Cleaned training messages.
    y_train : array-like of str
        Labels aligned with X_train ('ham' or 'spam').
    sample_size : int, default 500
        Number of ham messages to analyze.

    Returns
    -------
    dict[str, int]
        Mapping of word -> frequency within sampled ham messages.
    """
    ham_msgs = X_train[y_train == 'ham']
    limit = min(sample_size, len(ham_msgs))
    freq = {}
    for msg in ham_msgs[:limit]:
        for w in str(msg).split():
            if 2 < len(w) < 10:  # keep typical conversational tokens
                freq[w] = freq.get(w, 0) + 1
    return freq

wf_example = extract_ham_word_freq(X_train, y_train, sample_size=500)
print("[*] Example: extract_ham_word_freq")
print(f"  Ham messages sampled: {min(500, sum(y_train == 'ham'))}")
print(f"  Unique tokens found: {len(wf_example)}")
top5 = sorted(wf_example.items(), key=lambda x: (-x[1], x[0]))[:5]
for w, c in top5:
    print(f"    {w}: {c}")

[*] Example: extract_ham_word_freq
  Ham messages sampled: 500
  Unique tokens found: 2162
    you: 207
    the: 125
    and: 107
    that: 67
    for: 62


In [20]:
# This function mirrors how an attacker would gather data in the wild. It assumes access to some legitimate text, for example public forums or personal communications, and counts word usage. The split() method performs basic tokenization that matches our earlier preprocessing. The length filter (3..9) removes uninformative tokens while retaining short, conversational words that often indicate ham.

# The length constraints len(word) > 2 and len(word) < 10 filter out unhelpful tokens. Words with 2 or fewer characters (like "a", "I", "to") often appear in stop word lists and provide little discriminative value. Words longer than 10 characters are often URLs, phone numbers, or other artifacts rather than meaningful vocabulary. For instance, we'd exclude both "a" (too short) and "http://example.com" (too long), focusing on words like "meeting", "thanks", "tomorrow" that genuinely indicate legitimate communication.

In [21]:
# Selecting High-Frequency Words

def select_high_frequency_words(word_freq, max_words=100, min_freq=5):
    """
    Select the most frequent ham words above a minimum frequency.

    Parameters
    ----------
    word_freq : dict[str, int]
        Token frequency table for sampled ham messages.
    max_words : int, default 100
        Maximum number of words to return.
    min_freq : int, default 5
        Minimum frequency a word must meet to be considered.

    Returns
    -------
    list[str]
        Top words sorted by decreasing frequency then lexicographically.
    """
    sorted_by_freq = sorted(word_freq.items(), key=lambda x: (-x[1], x[0]))
    top = [w for w, c in sorted_by_freq if c > min_freq][:max_words]
    return top

top_words_example = select_high_frequency_words(wf_example, max_words=100, min_freq=5)
print("[*] Example: select_high_frequency_words")
print(f"  Selected top words: {len(top_words_example)} (min_freq=5)")
print("  First 10:", ", ".join(top_words_example[:10]))

[*] Example: select_high_frequency_words
  Selected top words: 100 (min_freq=5)
  First 10: you, the, and, that, for, your, are, have, but, not


In [22]:
# Merging Curated Conversational Terms
# Beyond frequency-based selection, we include carefully chosen conversational words that often appear in legitimate messages. These capture informal style, temporal references, and polite terms that are underrepresented in spam. This curated set hedges against sampling bias when the ham sample is small and against distribution shift in a black-box setting where we cannot inspect the model’s internal vocabulary.

def merge_with_curated(top_words, additional_candidates=None):
    """
    Merge data-driven top words with curated conversational candidates.

    Parameters
    ----------
    top_words : list[str]
        High-frequency ham words from the previous step.
    additional_candidates : list[str] | None
        Optional curated list to include regardless of frequency.

    Returns
    -------
    list[str]
        Deduplicated merged list (lexicographically ordered).
    """
    if additional_candidates is None:
        additional_candidates = [
            "ok", "cos", "ill", "thats", "later", "said", "ask", "didnt",
            "dont", "doing", "going", "come", "home", "tomorrow", "today", "sorry",
            "thanks", "yeah", "yes", "sure", "see", "tell", "know", "think",
        ]
    merged = set(top_words) | set(additional_candidates)
    return sorted(merged)

merged_example = merge_with_curated(top_words_example)
added = sorted(set(merged_example) - set(top_words_example))
print("[*] Example: merge_with_curated")
print(f"  Merged size: {len(merged_example)} | Added curated: {len(added)}")
print("  Sample added terms:", ", ".join(added[:5]))

[*] Example: merge_with_curated
  Merged size: 111 | Added curated: 11
  Sample added terms: ask, didnt, doing, ill, later


In [23]:
# Assembling and Verifying
def build_candidate_vocabulary(
    X_train,
    y_train,
    sample_size=500,
    max_words=100,
    min_freq=5,
    additional_candidates=None,
):
    """
    Build a candidate vocabulary for black-box discovery from ham messages.

    Parameters
    ----------
    X_train : array-like of str
        Cleaned training messages.
    y_train : array-like of str
        Labels aligned with X_train ('ham' or 'spam').
    sample_size : int, default 500
        Number of ham messages to analyze.
    max_words : int, default 100
        Maximum number of top frequent ham words to keep before merging extras.
    min_freq : int, default 5
        Minimum frequency threshold for inclusion from the ham corpus.
    additional_candidates : list[str] | None
        Optional curated conversational terms to include.

    Returns
    -------
    list[str]
        Deduplicated candidate words ordered by decreasing ham frequency,
        then lexicographically for stable ties.
    """
    word_freq = extract_ham_word_freq(X_train, y_train, sample_size=sample_size)
    top_words = select_high_frequency_words(word_freq, max_words=max_words, min_freq=min_freq)
    merged = merge_with_curated(top_words, additional_candidates=additional_candidates)

    # Stable final ordering driven by ham frequency, then lexical for ties
    def sort_key(w):
        return (-word_freq.get(w, 0), w)

    return sorted(merged, key=sort_key)

cv_example = build_candidate_vocabulary(X_train, y_train)
print("[*] Example: build_candidate_vocabulary")
print(f"  Candidates: {len(cv_example)}")
print("  First 10:", ", ".join(cv_example[:10]))

[*] Example: build_candidate_vocabulary
  Candidates: 111
  First 10: you, the, and, that, for, your, are, have, but, not


In [24]:
# Building the Candidate List
# Build candidate vocabulary for discovery
candidate_words = build_candidate_vocabulary(X_train, y_train)
print(f"[+] Testing {len(candidate_words)} candidate words extracted from ham messages")

[+] Testing 111 candidate words extracted from ham messages


In [25]:
# Query Management Framework
# The black-box attack must carefully manage its query budget across discovery and exploitation phases. Establish the framework for this management:

# Budget Allocation Strategy
# We'll reserve portions of the budget for different phases. First we implement a helper that returns a 40-40-20 split across exploration, exploitation, and combination, using the remainder for the last phase so totals always match the input. Then we print an example allocation for the current query_budget

def estimate_budget_allocation(total_budget):
    """
    Estimate allocation across exploration, exploitation, and combination.

    Parameters
    ----------
    total_budget : int
        Total query budget available for discovery.

    Returns
    -------
    dict
        Mapping phase -> integer number of queries that sums to `total_budget`.
    """
    explore = int(0.4 * total_budget)
    exploit = int(0.4 * total_budget)
    combine = total_budget - explore - exploit  # absorb rounding
    return {
        'exploration': explore,
        'exploitation': exploit,
        'combination': combine,
    }

# Quick demo for budget allocation
allocation = estimate_budget_allocation(query_budget)
print("\n[*] Budget allocation:")
for phase, budget in allocation.items():
    print(f"  {phase:12}: {budget:4d} queries")
print(f"  Total: {sum(allocation.values())} / {query_budget}")


[*] Budget allocation:
  exploration :  400 queries
  exploitation:  400 queries
  combination :  200 queries
  Total: 1000 / 1000


In [26]:
# Now we prepare inputs for Phase 1 by selecting a diverse subset of spam messages and shuffling candidates to avoid ordering bias:

# Discovery phase - test word effectiveness
word_scores = {}
spam_test_messages = X_test[y_test == 'spam']
test_spam_samples = spam_test_messages[:50]

# Test in batches to be more efficient
print(f"[*] Discovery phase: testing {len(candidate_words)} candidates...")

# Randomly sample candidates and messages for better coverage
np.random.shuffle(candidate_words)
np.random.shuffle(test_spam_samples)

[*] Discovery phase: testing 111 candidates...


/tmp/ipykernel_68912/3608068180.py:13: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(test_spam_samples)


In [27]:
# Black-Box Adaptive Discovery Methods
# We can employ adaptive learning techniques to efficiently discover effective words within query constraints. To this end, we'll implement epsilon-greedy selection and exponential moving averages to balance exploration of new candidates with exploitation of proven performers.

# Adaptive Scoring Functions

# Initializing the Scorer

def initialize_adaptive_scorer():
    """Initialize adaptive scoring data structures"""
    return {
        'word_scores': {},      # Maps word -> effectiveness score
        'word_counts': {},      # Maps word -> number of times tested
        'exploration_rate': 0.2  # 20% exploration for discovery phase
    }

In [28]:
# Epsilon-Greedy Word Selection
# Now let's implement the epsilon-greedy algorithm, a standard approach from multi-armed bandit theory. This algorithm balances the exploration‑exploitation tradeoff by occasionally taking risks to discover better options:

def epsilon_greedy_select(scorer, available_words):
    """Select word using epsilon-greedy strategy

    Parameters:
        scorer (dict): Adaptive scorer state
        available_words (list): Candidate words to choose from

    Returns:
        str: Selected word for testing
    """
    import random

    if random.random() < scorer['exploration_rate']:
        # Exploration: try untested or rarely tested words
        untested = [w for w in available_words if w not in scorer['word_counts']]
        if untested:
            return random.choice(untested)
        else:
            # Choose least tested word
            return min(available_words,
                      key=lambda w: scorer['word_counts'].get(w, 0))
    # Exploitation Strategy
    else:
        # Exploitation: choose best performing word
        return max(available_words,
                  key=lambda w: scorer['word_scores'].get(w, 0))

In [29]:
# Score Updates with Exponential Moving Average

def update_word_score(scorer, word, impact, alpha=0.3):
    """Update word score using exponential moving average

    Parameters:
        scorer (dict): Adaptive scorer state
        word (str): Word being scored
        impact (float): Observed reduction in spam probability
        alpha (float): Learning rate
    """
    if word not in scorer['word_scores']:
        scorer['word_scores'][word] = impact
        scorer['word_counts'][word] = 1
    else:
        # Exponential moving average
        old_score = scorer['word_scores'][word]
        scorer['word_scores'][word] = (1 - alpha) * old_score + alpha * impact
        scorer['word_counts'][word] += 1

In [30]:
# Combinatorial Search for Synergies
# Systematic Combination Testing

def discover_word_combinations(message, test_words, max_size=3):
    """Discover effective word combinations through systematic search

    Parameters:
        message (str): Target spam message
        test_words (list): Promising words to test
        max_size (int): Maximum combination size

    Returns:
        dict: Mapping of word combinations to effectiveness scores
    """
    from itertools import combinations

    combination_scores = {}
    message_vec = vectorizer.transform([message])
    message_score = classifier.predict_proba(message_vec)[0][1]

    # Test individual words first
    for word in test_words[:20]:
        test_message = message + " " + word
        test_vec = vectorizer.transform([test_message])
        score = classifier.predict_proba(test_vec)[0][1]
        impact = message_score - score
        combination_scores[(word,)] = impact
        # Discovering Pairwise Synergy
    # Test pairs for synergy
    if max_size >= 2:
        for word1, word2 in combinations(test_words[:15], 2):
            test_message = message + " " + word1 + " " + word2
            test_vec = vectorizer.transform([test_message])
            score = classifier.predict_proba(test_vec)[0][1]

            # Calculate synergy
            individual_impact = combination_scores.get((word1,), 0) + combination_scores.get((word2,), 0)
            actual_impact = message_score - score
            synergy = actual_impact - individual_impact

            if synergy > 0:  # Positive synergy detected
                combination_scores[(word1, word2)] = actual_impact
            # Building Triplet Combinations
    # Test triplets for top pairs
    if max_size >= 3:
        top_pairs = sorted(
            [(k, v) for k, v in combination_scores.items() if len(k) == 2],
            key=lambda x: x[1], reverse=True
        )[:5]

        for pair, pair_score in top_pairs:
            for word in test_words[:10]:
                if word not in pair:
                    triplet = tuple(sorted(pair + (word,)))
                    test_message = message + " " + " ".join(triplet)
                    test_vec = vectorizer.transform([test_message])
                    score = classifier.predict_proba(test_vec)[0][1]
                    combination_scores[triplet] = message_score - score

    return combination_scores

In [31]:
# Black-Box Discovery Algorithm

# Three-Phase Discovery Function

# Algorithm Initialization
def three_phase_discovery(spam_messages, candidate_words, budget=1000):
    """Three-phase discovery: exploration, exploitation, combination

    Parameters:
        spam_messages (list): Target spam messages
        candidate_words (list): Vocabulary to test
        budget (int): Total query budget

    Returns:
        tuple: (discovered_words, combination_scores, queries_used)
    """
    scorer = initialize_adaptive_scorer()
    queries_used = 0

    # Allocate budgets using 40-40-20 split strategy
    allocation = estimate_budget_allocation(budget)
    exploration_budget = allocation['exploration']
    exploitation_budget = allocation['exploitation']
    combination_budget = allocation['combination']
    # Phase 1: Broad Exploration
    # Phase 1: Broad exploration (allocated budget)
    print(f"[*] Phase 1: Exploration (budget: {exploration_budget} queries)")

    p1_marks = {
        max(1, int(0.25 * exploration_budget)),
        max(1, int(0.50 * exploration_budget)),
        max(1, int(0.75 * exploration_budget)),
    }
    p1_reported = set()

    # Select a message and a candidate word
    test_message = random.choice(spam_messages)
    word = epsilon_greedy_select(scorer, candidate_words)
    # Measuring Impact With Two Queries
    # Baseline and augmented spam probabilities
    vec_orig = vectorizer.transform([test_message])
    prob_orig = classifier.predict_proba(vec_orig)[0][1]  # spam prob

    vec_aug = vectorizer.transform([test_message + " " + word])
    prob_aug = classifier.predict_proba(vec_aug)[0][1]

    impact = prob_orig - prob_aug

    # Update running score and consume query budget
    update_word_score(scorer, word, impact)
    queries_used += 2

    # Optional milestone report
    if queries_used in p1_marks and queries_used not in p1_reported:
        top3 = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)[:3]
        print(
            f"  [P1 {queries_used}/{exploration_budget}] "
            f"tested_words={len(scorer['word_scores'])} | "
            f"top3=" + ", ".join(f"{w}:{s:.3f}" for w, s in top3)
        )
        p1_reported.add(queries_used)
        
    while queries_used < exploration_budget and len(candidate_words) > 0:
        # Select inputs
        test_message = random.choice(spam_messages)
        word = epsilon_greedy_select(scorer, candidate_words)

        # Measure impact with two queries
        vec_orig = vectorizer.transform([test_message])
        prob_orig = classifier.predict_proba(vec_orig)[0][1]
        vec_aug = vectorizer.transform([test_message + " " + word])
        prob_aug = classifier.predict_proba(vec_aug)[0][1]
        impact = prob_orig - prob_aug

        # Update score and account for budget
        update_word_score(scorer, word, impact)
        queries_used += 2

        # Milestone report
        if queries_used in p1_marks and queries_used not in p1_reported:
            top3 = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)[:3]
            print(
                f"  [P1 {queries_used}/{exploration_budget}] "
                f"tested_words={len(scorer['word_scores'])} | "
                f"top3=" + ", ".join(f"{w}:{s:.3f}" for w, s in top3)
            )
            p1_reported.add(queries_used)

# Exploration Results
    print(f"[+] Exploration complete. Queries: {queries_used}, Words tested: {len(scorer['word_scores'])}")
    top5 = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)[:5]
    if top5:
        print("  Top5 after exploration:")
        for w, s in top5:
            print(f"    {w:12} | score: {s:.3f}")

    # Phase 2: Focused exploitation
    scorer['exploration_rate'] = 0.1  # Reduce exploration

    # Get top words for exploitation
    top_words = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)[:30]
    top_word_list = [w for w, _ in top_words]

    print(f"\n[*] Phase 2: Exploitation (budget: {exploitation_budget} queries)")
    initial_queries = queries_used
    p2_mid = initial_queries + max(1, exploitation_budget // 2)

    # Concentrated Testing
    while queries_used < initial_queries + exploitation_budget and len(top_word_list) > 0:
        test_message = random.choice(spam_messages[:20])  # Focus on fewer messages
        word = random.choice(top_word_list[:15])  # Focus on best words

        vec_orig = vectorizer.transform([test_message])
        prob_orig = classifier.predict_proba(vec_orig)[0][1]

        vec_aug = vectorizer.transform([test_message + " " + word])
        prob_aug = classifier.predict_proba(vec_aug)[0][1]

        impact = prob_orig - prob_aug
        update_word_score(scorer, word, impact)
        queries_used += 2

    print(f"[+] Exploitation complete. Total queries: {queries_used}")

    # Phase 3: Combination discovery (allocated budget)
    remaining_combo = combination_budget
    print(f"\n[*] Phase 3: Combination search (budget: {remaining_combo} queries)")

    best_combinations = {}
    combos_tested = 0

    # Putting the search together
    if remaining_combo > 50:  # Need minimum queries for combinations
        for i in range(min(3, len(spam_messages))):
            if queries_used >= budget or remaining_combo <= 0:
                break

            test_msg = spam_messages[i]
            combos = discover_word_combinations(test_msg, top_word_list[:20], max_size=3)

            # Track best combinations across messages
            for combo, score in combos.items():
                if combo not in best_combinations or score > best_combinations[combo]:
                    best_combinations[combo] = score

            # Account for queries (~2 per combination) while respecting the budget
            to_add = min(remaining_combo, len(combos) * 2)
            queries_used += to_add
            remaining_combo -= to_add
            combos_tested += len(combos)

            # Midpoint snapshot
            if combination_budget > 0 and remaining_combo <= combination_budget // 2 and best_combinations:
                best = max(best_combinations.items(), key=lambda x: x[1])
                print(
                    f"  [P3 mid ~{combination_budget - remaining_combo}/{combination_budget}] "
                    f"combos_tested={combos_tested} | best={' + '.join(best[0])}:{best[1]:.3f}"
                )

            if remaining_combo <= 0:
                break

    # Final Results Compilation
    print(f"[+] Combination search complete. Total queries: {queries_used}")

    # Return final results
    final_words = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)
    return final_words, best_combinations, queries_used

In [33]:
discovered_words, combination_scores, total_queries = three_phase_discovery(
    test_spam_samples, candidate_words, budget=query_budget
)

[*] Phase 1: Exploration (budget: 400 queries)
  [P1 100/400] tested_words=13 | top3=dont:0.074, thanks:0.024, said:0.000
  [P1 200/400] tested_words=20 | top3=dont:0.115, thanks:0.001, thats:0.000
  [P1 300/400] tested_words=30 | top3=tell:0.120, yeah:0.036, dont:0.001
[+] Exploration complete. Queries: 400, Words tested: 40
  Top5 after exploration:
    yeah         | score: 0.070
    tell         | score: 0.019
    dont         | score: 0.001
    thanks       | score: 0.001
    know         | score: 0.000

[*] Phase 2: Exploitation (budget: 400 queries)
[+] Exploitation complete. Total queries: 800

[*] Phase 3: Combination search (budget: 200 queries)
  [P3 mid ~200/200] combos_tested=146 | best=feel + said + yeah:0.000
[+] Combination search complete. Total queries: 1000


In [35]:
# Black-Box Attack Implementation
# The white-box attack had perfect vision: direct access to probability tables, goodness scores computed from internal parameters, optimal word selection from complete model knowledge. Now we operate blind. No probability tables. No internal parameters. Just a query interface that returns predictions. Can we still identify effective good words? Can we still achieve high evasion rates? The three-phase discovery algorithm answers both questions affirmatively.

# Discovery Setup and Budget Allocation

print("\n[*] Using three-phase discovery algorithm...")

# Build candidate vocabulary
candidate_words = build_candidate_vocabulary(X_train, y_train)
print(f"[+] Built vocabulary of {len(candidate_words)} candidate words")


[*] Using three-phase discovery algorithm...
[+] Built vocabulary of 111 candidate words


In [36]:
# Show budget allocation
allocation = estimate_budget_allocation(query_budget)
print(f"\n[*] Budget allocation:")
for phase, budget in allocation.items():
    print(f"    {phase:12}: {budget:4d} queries")



[*] Budget allocation:
    exploration :  400 queries
    exploitation:  400 queries
    combination :  200 queries


In [37]:
# Running Three-Phase Discovery
# Run three-phase discovery
discovered_words, combination_scores, total_queries = three_phase_discovery(
    spam_test_messages[:50],
    candidate_words,
    budget=query_budget
)

print(f"\n[+] Discovery complete. Total queries used: {total_queries}/{query_budget}")
print(f"[+] Top 10 discovered words:")
for word, score in discovered_words[:10]:
    print(f"    {word:10} | impact: {score:.3f}")

[*] Phase 1: Exploration (budget: 400 queries)
  [P1 100/400] tested_words=15 | top3=sure:0.042, love:0.037, ill:0.036
  [P1 200/400] tested_words=24 | top3=i'll:0.012, day.:0.007, love:0.006
  [P1 300/400] tested_words=39 | top3=day.:0.002, sure:0.002, love:0.002
[+] Exploration complete. Queries: 400, Words tested: 52
  Top5 after exploration:
    like         | score: 0.019
    going        | score: 0.003
    ill          | score: 0.003
    i'll         | score: 0.002
    ask          | score: 0.002

[*] Phase 2: Exploitation (budget: 400 queries)
[+] Exploitation complete. Total queries: 800

[*] Phase 3: Combination search (budget: 200 queries)
  [P3 mid ~200/200] combos_tested=165 | best=... + ask + sure:0.000
[+] Combination search complete. Total queries: 1000

[+] Discovery complete. Total queries used: 1000/1000
[+] Top 10 discovered words:
    ill        | impact: 0.227
    sure       | impact: 0.197
    ask        | impact: 0.110
    like       | impact: 0.105
    ...      

In [38]:
# Display Combinations
if combination_scores:
    print(f"\n[+] Top 5 word combinations:")
    top_combos = sorted(combination_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    for combo, score in top_combos:
        combo_str = ', '.join(combo)
        print(f"    {combo_str:30} | synergy: {score:.3f}")

# Update queries_used for compatibility
queries_used = total_queries


[+] Top 5 word combinations:
    ..., ask, sure                 | synergy: 0.000
    ..., ask, going                | synergy: 0.000
    ..., ask, dun                  | synergy: 0.000
    ..., ask, gonna                | synergy: 0.000
    ..., ask, went                 | synergy: 0.000


In [40]:
# Testing Discovered Words
# Test discovered words
blackbox_results = []
test_counts = [0, 5, 10, 15, 20, 25, 30]  # Test with more words

for num_words in test_counts:
    if queries_used >= query_budget:
        break

    selected = [w for w, _ in discovered_words[:num_words]]
    evaded = 0
    tested = 0

    # Test on a different subset of spam messages
    eval_messages = spam_test_messages[30:50]  # Different messages from discovery

    # Measuring Evasion Success
    for msg in eval_messages:
        if queries_used >= query_budget:
            break

        aug = msg if num_words == 0 else msg + " " + " ".join(selected)
        vec = vectorizer.transform([aug])
        prob = classifier.predict_proba(vec)[0][1]  # spam prob
        queries_used += 1

        if prob < 0.5:  # evasion threshold
            evaded += 1
        tested += 1
    
    # After evaluating all messages for a given word count, we compute and record the evasion rate:
    if tested > 0:
        rate = (evaded / tested) * 100
        blackbox_results.append({'num_words': num_words, 'evasion_rate': rate})
        print(f"  Words: {num_words:2d} | Evasion: {rate:6.2f}% | Queries total: {queries_used}")

print(f"\n[+] Black-box attack complete. Total queries: {queries_used}/{query_budget}")


[+] Black-box attack complete. Total queries: 1000/1000
